# Merlin – Example Pipeline

This notebook walks through the full **Merlin** SBI pipeline for Euclid 3×2pt cosmology:

1. Load a config file and inspect settings
2. Build the `Simulator` and draw prior samples
3. Generate a fiducial observation and apply Cholesky whitening
4. Load a Zarr store and run PCA compression
5. Visualise the swyft dataset and parameter priors

> **Prerequisites**: install the package from the repo root with `pip install -e ..`, and ensure `cloelib`, `euclidlib`, and `swyft` are available in your environment.

In [1]:
import configparser
import numpy as np
import matplotlib.pyplot as plt

from merlin.params import PARAMS, COSMO_PARAMS, N_COSMO
from merlin.fisher import get_sigmas_bounds
from merlin.tracers import load_dndz
from merlin.simulator import Simulator
from merlin.preprocessing import (
    apply_cholesky_to_obs,
    load_or_precompute_cholesky,
    load_or_compute_pca,
    make_resampler,
)
from merlin.io import save_predictions, load_file

ModuleNotFoundError: No module named 'merlin'

## 1 · Load config

Edit `CONFIG_PATH` below to point to your `.ini` file.  
The cell prints all sections and the fiducial parameter values so you can verify the paths are correct.

In [ ]:
CONFIG_PATH = "../examples/old_config.ini"   # ← change to your config

config = configparser.ConfigParser()
config.read(CONFIG_PATH)

print("Sections found:", config.sections())
print("\n[FIDUCIAL VALUES]")
for k, v in config["FIDUCIAL VALUES"].items():
    print(f"  {k:30s} = {v}")

## 2 · Build the simulator

Load the auxiliary files (covariance matrix, ell grid, mean redshifts, n(z)) and instantiate `Simulator`.
The cell also prints the number of spectra per probe combination.

In [ ]:
fiducial   = [float(v) for v in config["FIDUCIAL VALUES"].values()]
covmat     = np.load(config["AUX FILES"]["covmat"])["Gauss"]
n_bins     = int(config["FINV"]["Nbin_z"])
ell_theory = np.load(config["AUX FILES"]["ell_file"])
zmean      = np.load(config["AUX FILES"]["zmean_file"])
dndz       = load_dndz(config["AUX FILES"]["nz_example"])

_, lower_bounds, upper_bounds = get_sigmas_bounds(
    fiducial,
    config["FINV"]["finv_file"],
    int(config["FINV"]["N_pars"]),
)

sim = Simulator(
    fiducial=fiducial,
    covmat=covmat,
    n_bins=n_bins,
    lower_bounds=lower_bounds,
    upper_bounds=upper_bounds,
    zmean=zmean,
    ell_theory=ell_theory,
    dndz=dndz,
)

print(f"n_bins        = {sim.n_bins}")
print(f"n_data        = {sim.n_data}  (WL + GGL + GCph combined)")
print(f"WL  spectra   = {len(sim.WL_keys)}")
print(f"GGL spectra   = {len(sim.GGL_keys)}")
print(f"GCph spectra  = {len(sim.GG_keys)}")
print(f"ell range     = [{ell_theory.min():.0f}, {ell_theory.max():.0f}]  ({len(ell_theory)} bins)")

## 3 · Draw prior samples

Draw a handful of parameter samples from the prior and compute the corresponding
$C_\ell$ data vectors to verify the forward model is working.

In [ ]:
N_test = 5
z_samples = sim.sample_z(shape=(N_test,))   # (N_test, N_params)

print("Cosmological parameter names:", COSMO_PARAMS)
print(f"\n{N_test} prior draws (cosmo only):")
print(z_samples[:, :N_COSMO].round(4))

fig, ax = plt.subplots(figsize=(8, 4))
for z in z_samples:
    cls = sim.get_sample_Cls(z)
    n_wl = len(sim.WL_keys)
    wl_cls = cls[:n_wl * len(ell_theory)].reshape(n_wl, len(ell_theory))
    ax.plot(ell_theory, wl_cls.T, lw=0.9, alpha=0.6)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$C_\ell^{\rm WL}$")
ax.set_title(f"WL auto-spectra for {N_test} prior draws")
plt.tight_layout()
plt.show()

## 4 · Generate a fiducial observation

`generate_observation()` runs the forward model at the fiducial cosmology and
draws a noise realisation from the covariance.  We then Cholesky-whiten both
signal and noise and build the noiseless version used for inference.

In [ ]:
obs = sim.generate_observation()

print("Observation keys :", list(obs.keys()))
print("C_ells shape     :", obs["C_ells"].shape)
print("noise  shape     :", obs["noise"].shape)

# Cholesky-whiten
oCells_chol, onoise_chol = apply_cholesky_to_obs(obs, sim.Lfid)

obs_chol_noiseless = {
    "C_ells": oCells_chol,
    "noise": np.zeros_like(onoise_chol),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ell_theory, obs["C_ells"].T, lw=0.5, alpha=0.4)
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel(r"$\ell$"); axes[0].set_ylabel(r"$C_\ell$")
axes[0].set_title("Raw fiducial $C_\\ell$s")

axes[1].plot(ell_theory, oCells_chol.T, lw=0.5, alpha=0.4)
axes[1].set_xscale("log")
axes[1].set_xlabel(r"$\ell$"); axes[1].set_ylabel("Whitened $C_\\ell$")
axes[1].set_title("Cholesky-whitened $C_\\ell$s")

plt.tight_layout()
plt.show()

## 5 · Load a Zarr store and apply PCA compression

Run `merlin-simulate config.ini` first to populate the store.
This cell loads the store, applies Cholesky whitening (or reads from cache), and
computes / loads the PCA projection matrix.

In [ ]:
import swyft

store_path = config["SIMULATION"]["store_path"]
store = swyft.ZarrStore(store_path).get_sample_store()

print(f"Store contains {len(store['z'])} simulations")
print("Store keys    :", list(store.keys()))

# Cholesky whitening (cached after first run)
cache_dir = config["STORES"].get("cache_dir", "cache")
Cells_chol, noise_chol = load_or_precompute_cholesky(store, sim.Lfid, cache_dir)
print(f"\nCells_chol : {Cells_chol.shape}")
print(f"noise_chol : {noise_chol.shape}")

# PCA compression
V_proj = load_or_compute_pca(Cells_chol, config)
print(f"\nV_proj     : {V_proj.shape}  →  {V_proj.shape[1]} modes retained")

## 6 · Visualise PCA variance and parameter priors

Check how many modes are needed to reach 99.9 % variance, and visualise the
prior distributions for the cosmological parameters.

In [ ]:
import torch

# ── PCA cumulative variance ────────────────────────────────────────────────
fCells = torch.from_numpy(Cells_chol.reshape(len(store["z"]), -1))
_, S, _ = torch.pca_lowrank(fCells, q=min(100, fCells.shape[1]), center=True)
cumvar = (S.cumsum(0) / S.sum()).numpy() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(np.arange(1, len(cumvar) + 1), cumvar, marker="o", ms=3)
axes[0].axhline(99.9, color="tomato", ls="--", label="99.9 %")
axes[0].set_xlabel("PCA modes")
axes[0].set_ylabel("Cumulative variance (%)")
axes[0].set_title("PCA compression")
axes[0].legend()

# ── Cosmological parameter priors ─────────────────────────────────────────
z_np = store["z"][:, :N_COSMO]
for i, name in enumerate(COSMO_PARAMS):
    axes[1].hist(z_np[:, i], bins=40, alpha=0.6, label=name, histtype="stepfilled")

axes[1].set_xlabel("Parameter value")
axes[1].set_ylabel("Count")
axes[1].set_title("Prior samples – cosmological parameters")
axes[1].legend(fontsize=8)

n_ret = V_proj.shape[1]
print(f"Retained {n_ret} modes → {cumvar[n_ret - 1]:.3f} % variance")

plt.tight_layout()
plt.show()

## Next steps

| Step | Command |
|------|---------|
| Generate observation | `merlin-generate-obs config.ini` |
| Run simulations (cluster) | `merlin-simulate config.ini` |
| Train network & infer | `merlin-train config.ini` |
| Visualise posteriors | Open `Merlin_results.ipynb` |

See the [README](../README.md) for full configuration options.